In [50]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os
import urllib.request
import zipfile
import numpy as np


print("TensorFlow versione:", tf.__version__)

TensorFlow versione: 2.20.0


In [51]:
# Scaricare i dataset
print("Scaricando training set...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/download.tensorflow.org/data/rps.zip",
    "rps.zip"
)

print("Scaricando test set...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip",
    "rps-test-set.zip"
)

print("Estraendo...")
with zipfile.ZipFile("rps.zip", "r") as z:
    z.extractall(".")
with zipfile.ZipFile("rps-test-set.zip", "r") as z:
    z.extractall(".")

print("Dataset pronti!")

Scaricando training set...
Scaricando test set...
Estraendo...
Dataset pronti!


In [52]:
# Percorsi delle cartelle
TRAIN_DIR = "./rps"
TEST_DIR  = "./rps-test-set"

# Generatore TRAINING con augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalizzazione
    rotation_range=40,       # Ruota fino a 40 gradi
    width_shift_range=0.2,   # Sposta orizzontalmente
    shear_range=0.2,         # Deforma
    horizontal_flip=True,    # Specchia
    fill_mode='nearest'      # Riempie i pixel vuoti
)

# Generatore TEST (solo normalizzazione, niente augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

# Collega i generatori alle cartelle
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(150, 150),  # Ridimensiona ogni immagine a 150x150
    batch_size=32,
    class_mode='categorical' # 3 classi: rock, paper, scissors
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)

print("Classi trovate:", train_generator.class_indices)

Found 2520 images belonging to 3 classes.
Found 372 images belonging to 3 classes.
Classi trovate: {'paper': 0, 'rock': 1, 'scissors': 2}


In [ ]:
# Alcune immagini per visualizzare il processo
sample_images, sample_labels = next(train_generator)

plt.figure(figsize=(10, 5))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(sample_images[i])
    plt.axis('off')
plt.suptitle("Esempi di immagini con Augmentation")
plt.show()

In [54]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([

    # --- BLOCCO 1 ---
    # 32 filtri cercano pattern semplici (bordi, linee)
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    # --- BLOCCO 2 ---
    # 64 filtri cercano pattern più complessi (curve, angoli)
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- BLOCCO 3 ---
    # 128 filtri cercano pattern ancora più complessi (forme di dita)
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- BLOCCO 4 ---
    # 128 filtri per affinare ulteriormente
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- PARTE FINALE ---
    Flatten(),           # Srotola tutto in una lista
    Dense(512, activation='relu'),  # 512 neuroni per "ragionare"
    Dense(3, activation='softmax')  # 3 neuroni = 3 classi finali
])

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_16 (Conv2D)              │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │     3,211,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,147 (13.18 MB)

 Trainable params: 3,454,147 (13.18 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
# Strategia di apprendimento
model.compile(
    optimizer='adam',                    # Algoritmo di ottimizzazione
    loss='categorical_crossentropy',     # Funzione di errore per 3+ classi
    metrics=['accuracy']                 # Monitora l'accuratezza durante il training
)

print("Modello compilato, pronto per il training!")

Modello compilato, pronto per il training!


In [56]:
from tensorflow.keras.callbacks import EarlyStopping

# EarlyStopping: interrompe il training se accuracy supera 98%
early_stop = EarlyStopping(
    monitor='accuracy',      # Controlla l'accuratezza sul training set
    patience=3,              # Aspetta 3 epoche prima di fermarsi
    verbose=1                # Stampa un messaggio quando si ferma
)

# Avvia il training!
history = model.fit(
    train_generator,         # Dati di training con augmentation
    epochs=20,               # Massimo 20 epoche
    validation_data=test_generator,   # Valuta su test set ad ogni epoca
    callbacks=[early_stop]   # Usa l'EarlyStopping
)

Epoch 1/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 33s 349ms/step - accuracy: 0.5968 - loss: 0.7961 - val_accuracy: 0.6640 - val_loss: 0.8341
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 25s 315ms/step - accuracy: 0.8587 - loss: 0.3554 - val_accuracy: 0.9704 - val_loss: 0.1285
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 25s 311ms/step - accuracy: 0.9599 - loss: 0.1238 - val_accuracy: 0.9597 - val_loss: 0.1012
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 25s 315ms/step - accuracy: 0.9802 - loss: 0.0543 - val_accuracy: 0.9516 - val_loss: 0.1123
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 25s 316ms/step - accuracy: 0.9770 - loss: 0.0704 - val_accuracy: 0.9032 - val_loss: 0.2708
Epoch 6/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 25s 319ms/step - accuracy: 0.9909 - loss: 0.0306 - val_accuracy: 0.9194 - val_loss: 0.1673
Epoch 7/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 307ms/step - accuracy: 0.9893 - loss: 0.0424 - val_accuracy: 0.9651 - val_loss: 0.0628
Epoch 8/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 306ms/step - accuracy: 0.9885 - loss: 0.0322 - val_accu

In [ ]:
# Visualizza l'andamento del training
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

# Grafico accuratezza
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Accuratezza nel tempo')

# Grafico loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Errore nel tempo')

plt.show()

In [58]:
# Valuta il modello sul dataset di test
test_loss, test_accuracy = model.evaluate(test_generator)

print(f"\nRisultati sul Test Set:")
print(f"Accuratezza: {test_accuracy*100:.2f}%")
print(f"Errore:      {test_loss:.4f}")

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - accuracy: 0.9812 - loss: 0.0699

Risultati sul Test Set:
Accuratezza: 98.12%
Errore:      0.0699


In [ ]:
# Prendi un batch di immagini dal test set
sample_images, sample_labels = next(test_generator)

# Fai le previsioni
predictions = model.predict(sample_images)

# Nomi delle classi
class_names = ['paper', 'rock', 'scissors']

# Mostra 9 immagini con previsione
plt.figure(figsize=(12, 12))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(sample_images[i])

    # Classe prevista e reale
    predicted = class_names[np.argmax(predictions[i])]
    actual = class_names[np.argmax(sample_labels[i])]
    confidence = np.max(predictions[i]) * 100

    # Verde se giusto, rosso se sbagliato
    color = 'green' if predicted == actual else 'red'
    plt.title(f"Previsto: {predicted} ({confidence:.0f}%)\nReale: {actual}",
              color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io

# Codice JavaScript per aprire la webcam
def take_photo():
    js = Javascript('''
        async function takePhoto() {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Scatta foto';
            capture.style.fontSize = '20px';
            capture.style.padding = '10px';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            video.style.width = '300px';
            div.appendChild(video);

            document.body.appendChild(div);

            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(track => track.stop());
            div.remove();
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        takePhoto()
    ''')
    display(js)
    data = eval_js('takePhoto()')
    binary = b64decode(data.split(',')[1])
    return Image.open(io.BytesIO(binary))

# Scatta la foto
print("Clicca il pulsante per scattare la foto!")
img = take_photo()

# Prepara l'immagine per la rete (stesso preprocessing del training)
img_resized = img.resize((150, 150))
img_array = np.array(img_resized) / 255.0
img_array = np.expand_dims(img_array, axis=0)  # aggiunge dimensione batch

# Fai la previsione
prediction = model.predict(img_array)
class_names = ['paper', 'rock', 'scissors']
predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction) * 100

# Mostra risultato
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f"Previsto: {predicted_class}\nConfidenza: {confidence:.1f}%",
          fontsize=14)
plt.axis('off')
plt.show()

print(f"\nRisultato: {predicted_class} con {confidence:.1f}% di confidenza")

In [61]:
print("=" * 50)
print("ANALISI DEL DOMAIN GAP")
print("=" * 50)
print(f"\nAccuratezza su immagini CGI (test set): {test_accuracy*100:.2f}%")
print(f"\nRisultato foto reale:")
print(f"   Classe prevista: {predicted_class}")
print(f"   Confidenza:      {confidence:.1f}%")
print(f"\nAnalisi:")

if confidence > 90:
    print("   Il modello generalizza bene anche su foto reali!")
    print("   Il domain gap è minimo.")
elif confidence > 70:
    print("   Il modello funziona ma con meno sicurezza su foto reali.")
    print("   Il domain gap è moderato — le immagini CGI e reali")
    print("   hanno differenze di sfondo e illuminazione.")
else:
    print("   Il modello fatica sulle foto reali.")
    print("   Il domain gap è significativo — il modello è troppo")
    print("   abituato agli sfondi bianchi del dataset CGI.")

ANALISI DEL DOMAIN GAP

Accuratezza su immagini CGI (test set): 98.12%

Risultato foto reale:
   Classe prevista: paper
   Confidenza:      100.0%

Analisi:
   Il modello generalizza bene anche su foto reali!
   Il domain gap è minimo.
